In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv(
    '/content/drive/MyDrive/SpamDetection/emails.csv',
    engine='python',
    on_bad_lines='skip'
)

df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...


In [ ]:
print(df["message"].iloc[0:20])

0     Message-ID: <18782981.1075855378110.JavaMail.e...
1     Message-ID: <15464986.1075855378456.JavaMail.e...
2     Message-ID: <24216240.1075855687451.JavaMail.e...
3     Message-ID: <13505866.1075863688222.JavaMail.e...
4     Message-ID: <30922949.1075863688243.JavaMail.e...
5     Message-ID: <30965995.1075863688265.JavaMail.e...
6     Message-ID: <16254169.1075863688286.JavaMail.e...
7     Message-ID: <17189699.1075863688308.JavaMail.e...
8     Message-ID: <20641191.1075855687472.JavaMail.e...
9     Message-ID: <30795301.1075855687494.JavaMail.e...
10    Message-ID: <33076797.1075855687515.JavaMail.e...
11    Message-ID: <25459584.1075855687536.JavaMail.e...
12    Message-ID: <13116875.1075855687561.JavaMail.e...
13    Message-ID: <2707340.1075855687584.JavaMail.ev...
14    Message-ID: <2465689.1075855687605.JavaMail.ev...
15    Message-ID: <1115198.1075855687626.JavaMail.ev...
16    Message-ID: <19773657.1075855687649.JavaMail.e...
17    Message-ID: <7391389.1075855378477.JavaMai

In [ ]:
from email import message_from_string
import pandas as pd

def parse_email(raw_email):

    if pd.isna(raw_email):
        return pd.Series({
            "message_id" : None,
            "date" : None,
            "from" : None,
            "to" : None,
            "subject" : None,
            "body" : None
        })

    msg = message_from_string(raw_email)
    return pd.Series({
        "message_id" : msg.get("Message-ID"),
        "date" : msg.get("Date"),
        "from" : msg.get("From"),
        "to" : msg.get("To"),
        "subject" : msg.get("Subject"),
        "body" : msg.get_payload()
    })


In [ ]:
structured_df = df["message"].apply(parse_email)

structured_df.head(1000)

,message_id,date,from,to,subject,body
0,<18782981.1075855378110.JavaMail.evans@thyme>,"Mon, 14 May 2001 16:39:00 -0700 (PDT)",phillip.allen@enron.com,tim.belden@enron.com,,Here is our forecast\n\n
1,<15464986.1075855378456.JavaMail.evans@thyme>,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",phillip.allen@enron.com,john.lavorato@enron.com,Re:,Traveling to have a business meeting takes the...
2,<24216240.1075855687451.JavaMail.evans@thyme>,"Wed, 18 Oct 2000 03:00:00 -0700 (PDT)",phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,test successful. way to go!!!
3,<13505866.1075863688222.JavaMail.evans@thyme>,"Mon, 23 Oct 2000 06:13:00 -0700 (PDT)",phillip.allen@enron.com,randall.gay@enron.com,,"Randy,\n\n Can you send me a schedule of the s..."
4,<30922949.1075863688243.JavaMail.evans@thyme>,"Thu, 31 Aug 2000 05:07:00 -0700 (PDT)",phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Let's shoot for Tuesday at 11:45.
...,...,...,...,...,...,...
995,<20430828.1075855696096.JavaMail.evans@thyme>,"Mon, 19 Mar 2001 01:36:00 -0800 (PST)",phillip.allen@enron.com,jacquestc@aol.com,,"Jacques,\n\nStill trying to close the loop on ..."
996,<18425275.1075855696118.JavaMail.evans@thyme>,"Mon, 19 Mar 2001 00:45:00 -0800 (PST)",phillip.allen@enron.com,llewter@austin.rr.com,Re: Buyout,"Larrry,\n\nI realize you are disappointed abou..."
997,<24036204.1075855666506.JavaMail.evans@thyme>,"Wed, 6 Dec 2000 08:04:00 -0800 (PST)",phillip.allen@enron.com,pallen70@hotmail.com,,---------------------- Forwarded by Phillip K ...
998,<33307764.1075855696139.JavaMail.evans@thyme>,"Fri, 16 Mar 2001 04:28:00 -0800 (PST)",phillip.allen@enron.com,jacquestc@aol.com,,"Jacques,\n\nI think we reached an agreement wi..."


For SPAM detection **combine message and body column** because both subject and body contains text which can we identified as malicious

In [6]:
structured_df["text"] = structured_df["subject"].fillna("") +" "+ structured_df["body"].fillna("")      ## fillna will replace NULL values with Empty string

structured_df['text'].head()

,text
0,Here is our forecast\n\n
1,Re: Traveling to have a business meeting takes...
2,Re: test test successful. way to go!!!
3,"Randy,\n\n Can you send me a schedule of the ..."
4,Re: Hello Let's shoot for Tuesday at 11:45.


Clean the text

In [7]:
import re

def clean_text(text):
  text = str(text).lower()
  text = re.sub(r'http\S+', '', text)                   # if there are any links starting with http or http followed by s without any space, replace with empty string
  text = re.sub(r'[^a-zA-Z\s]', " ", text)              # If not (^) a-z or A-Z or whitespace (\s), replace with empty string
  text = re.sub(r'\s+', " ", text)                      # If more than one spaces or empty lines, will be replaced with single whitespace
  return text.strip()

structured_df["text"] = structured_df["text"].apply(clean_text)
structured_df["text"].head()

,text
0,here is our forecast
1,re traveling to have a business meeting takes ...
2,re test test successful way to go
3,randy can you send me a schedule of the salary...
4,re hello let s shoot for tuesday at


Add labels to

In [13]:
# add labels to

structured_df["label"] = 0

structured_df[["text","label"]].head()

,text,label
0,here is our forecast,0
1,re traveling to have a business meeting takes ...,0
2,re test test successful way to go,0
3,randy can you send me a schedule of the salary...,0
4,re hello let s shoot for tuesday at,0


In [14]:
# save this structured dataset to new dataset

structured_df.to_csv(
    "/content/drive/MyDrive/SpamDetection/enron_processed.csv",
    index=False
)

GET ENRON SPAM DATASET

In [15]:
spam_folder = "/content/drive/MyDrive/SpamDetection/20030228_spam/spam"

spam_records = []

In [17]:
import os
import pandas as pd
for filename in os.listdir(spam_folder):

    filepath = os.path.join(spam_folder, filename)

    try:
        with open(filepath, "r", encoding="latin-1") as f:
            raw_email = f.read()

        msg = message_from_string(raw_email)

        subject = msg.get("Subject", "")

        body = msg.get_payload()

        spam_records.append({
            "subject": subject,
            "body": body,
            "label": 1
        })

    except Exception as e:
        print(f"Error reading {filename}: {e}")

spam_df = pd.DataFrame(spam_records)

spam_df.head()

,subject,body,label
0,Fw: Re: Account For Cum Shots To: zzzz@spamass...,#############################################...,1
1,[ILUG] Discount Fags,Dear Sir / Madam\n\nIf you are fed up of being...,1
2,"Fw: PROTECT YOUR COMPUTER,YOU NEED SYSTEMWORKS...","<html>\n\n<head>\n<meta http-equiv=3D""Content-...",1
3,Low Price Tobacco,Dear Sir / Madam\n\nIf you are fed up of being...,1
4,WebDesignHQ Newsletter,We thought you may be interested in our new so...,1


In [35]:
spam_df["text"] = (
    spam_df["subject"].astype(str) +
    " " +
    spam_df["body"].astype(str)
)

In [37]:
spam_df.head()

,subject,body,label,text
0,Fw: Re: Account For Cum Shots To: zzzz@spamass...,#############################################...,1,Fw: Re: Account For Cum Shots To: zzzz@spamass...
1,[ILUG] Discount Fags,Dear Sir / Madam\n\nIf you are fed up of being...,1,[ILUG] Discount Fags Dear Sir / Madam\n\nIf yo...
2,"Fw: PROTECT YOUR COMPUTER,YOU NEED SYSTEMWORKS...","<html>\n\n<head>\n<meta http-equiv=3D""Content-...",1,"Fw: PROTECT YOUR COMPUTER,YOU NEED SYSTEMWORKS..."
3,Low Price Tobacco,Dear Sir / Madam\n\nIf you are fed up of being...,1,Low Price Tobacco Dear Sir / Madam\n\nIf you a...
4,WebDesignHQ Newsletter,We thought you may be interested in our new so...,1,WebDesignHQ Newsletter We thought you may be i...


In [38]:
import re

def clean_text(text):
  text = str(text).lower()
  text = re.sub(r'http\S+', '', text)                   # if there are any links starting with http or http followed by s without any space, replace with empty string
  text = re.sub(r'[^a-zA-Z\s]', " ", text)              # If not (^) a-z or A-Z or whitespace (\s), replace with empty string
  text = re.sub(r'\s+', " ", text)                      # If more than one spaces or empty lines, will be replaced with single whitespace
  return text.strip()

spam_df["text"] = spam_df["text"].apply(clean_text)
spam_df["text"].head()

,text
0,fw re account for cum shots to zzzz spamassass...
1,ilug discount fags dear sir madam if you are f...
2,fw protect your computer you need systemworks ...
3,low price tobacco dear sir madam if you are fe...
4,webdesignhq newsletter we thought you may be i...


In [39]:
enron_df = structured_df[["text", "label"]]

final_df = pd.concat(
    [enron_df, spam_df[["text", "label"]]],
    ignore_index=True
)

print(final_df.shape)

(843723, 2)


In [40]:
final_df["label"].value_counts()

,count
label,
0,843222
1,501


In [42]:
ham_df = final_df[final_df["label"] == 0].sample(n=500, random_state=42)

spam_df = final_df[final_df["label"] == 1]

balanced_df = pd.concat([ham_df, spam_df], ignore_index = True)

In [43]:
balanced_df["label"].value_counts()

,count
label,
1,501
0,500


In [44]:
balanced_df = balanced_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [45]:
balanced_df.to_csv(
    "/content/drive/MyDrive/SpamDetection/balanced_spam_dataset.csv",
    index=False
)

MODEL TRANING

In [47]:
from sklearn.model_selection import train_test_split
x = balanced_df["text"]
y = balanced_df["label"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [49]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(
    stop_words = "english",
    max_features = 5000
)

x_train_tfidf = vectorizer.fit_transform(x_train)
x_test_tfidf = vectorizer.transform(x_test)

Logistic Regression

In [51]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression()

lr_model.fit(x_train_tfidf, y_train)

LogisticRegression()

Naive Bayes

In [53]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()

nb_model.fit(x_train_tfidf, y_train)

MultinomialNB()

In [55]:
lr_pred = lr_model.predict(x_test_tfidf)
nb_pred = nb_model.predict(x_test_tfidf)

In [56]:
from sklearn.metrics import classification_report

print("Logistic Regression")
print(classification_report(y_test, lr_pred))

print("\nNaive Bayes")
print(classification_report(y_test, nb_pred))

Logistic Regression
              precision    recall  f1-score   support

           0       0.95      1.00      0.98       100
           1       1.00      0.95      0.97       101

    accuracy                           0.98       201
   macro avg       0.98      0.98      0.98       201
weighted avg       0.98      0.98      0.98       201


Naive Bayes
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       100
           1       0.99      1.00      1.00       101

    accuracy                           1.00       201
   macro avg       1.00      0.99      1.00       201
weighted avg       1.00      1.00      1.00       201



In [57]:
def predict_spam(email_text):
    text = clean_text(email_text)
    vector = vectorizer.transform([text])
    prediction = nb_model.predict(vector)[0]

    return "Spam" if prediction == 1 else "Ham"

In [58]:
predict_spam("Congratulations! You have won a free iPhone. Click now!")

'Spam'

In [59]:
import joblib

joblib.dump(nb_model, "spam_detector.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")


['tfidf_vectorizer.pkl']

In [60]:
pip install fastapi uvicorn